In [5]:
import sys, os, importlib
# Ensure repository root is on sys.path so sam_3d_body is importable
repo_root = os.path.dirname(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import sam_3d_body.models.meta_arch.sam3d_body as sam3d_body_module
importlib.reload(sam3d_body_module)
print('Reloaded sam3d_body module')

Reloaded sam3d_body module


In [5]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## 1. Imports and Model Loading

In [6]:
# !pip install ipywidgets
# from huggingface_hub import login
# login()

In [1]:
import sys
import os

parent_dir = os.path.dirname(os.getcwd())
sys.path.insert(0, parent_dir)

# Import helper utilities from the notebook package
from notebook.utils import (
    setup_sam_3d_body, setup_visualizer,
    visualize_2d_results, visualize_3d_mesh, save_mesh_results,
    display_results_grid, process_image_with_mask
)

# Set up SAM 3D Body estimator
# estimator = setup_sam_3d_body(hf_repo_id="facebook/sam-3d-body-dinov3")
from sam_3d_body import load_sam_3d_body_hf, SAM3DBodyEstimator
sam3d_model, estimator_cfg = load_sam_3d_body_hf("facebook/sam-3d-body-dinov3", device="cpu")
estimator = SAM3DBodyEstimator(sam_3d_body_model=sam3d_model, model_cfg=estimator_cfg)

# Set up visualizer
visualizer = setup_visualizer()

c:\Users\16047\Documents\GitHub\sam-3d-body\sam_3d_body\models\heads\mhr_head.py:33: UserWarning: Momentum is not enabled
  warnings.warn("Momentum is not enabled")


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading SAM 3D Body model...


Using cache found in C:\Users\16047/.cache\torch\hub\facebookresearch_dinov3_main
Ignored kwargs: {'drop_path': 0.1}
Ignored kwargs: {'drop_path': 0.1}
The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.encoder.mask_token, head_pose.hand_pose_comps_ori, head_pose.mhr.face_expressions_model.shape_vectors, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_indices, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_weight, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.2.weight, head_pose.mhr.character_torch.skeleton.joint_translation_offsets, head_pose.mhr.character_torch.skeleton.joint_prerotations, head_pose.mhr.character_torch.skeleton.pmi, head_pose.mhr.character_torch.skeleton.joint_parents, head_pose.mhr.character_torch.mesh.rest_vertices, head_pose.mhr.character_torch.mesh.faces, head_pose.mhr.character_torch.mesh.texcoords, head_pose.mhr.character_torch.mesh.texcoord_faces, head_pose.mhr.chara

No human detector is used...
Mask-condition inference is not supported...
No FOV estimator... Using the default FOV!


## 2. Process Image and Get Outputs

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

# Load and process the image
image_path = "images/dancing.jpg"  # Relative to notebook folder
img_cv2 = cv2.imread(image_path)
if img_cv2 is None:
    raise FileNotFoundError(f"Image not found at {image_path}. Ensure the file exists.")
img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)

print("Processing image with SAM 3D Body via process_one_image...")

# Run inference using the estimator high-level API
outputs = estimator.process_one_image(image_path)

print(f"Number of people detected: {len(outputs)}")
print(f"Output keys for first person: {list(outputs[0].keys()) if outputs else 'No people detected'}")

# Display the original image
plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.axis('off')
plt.title('Original Image')
plt.show()

Processing image with SAM 3D Body via process_one_image...


## 3. 2D Visualization - Keypoints and Bounding Boxes

In [ ]:
# Visualize 2D results using utils
if outputs:
    vis_results = visualize_2d_results(img_cv2, outputs, visualizer)
    
    # Display results using grid function
    titles = [f'Person {i} - 2D Keypoints & BBox' for i in range(len(vis_results))]
    display_results_grid(vis_results, titles, figsize_per_image=(6, 6))
else:
    print("No people detected in the image")

## 4. 3D Mesh Visualization - Overlay and Side View

In [ ]:
if outputs:
    mesh_results = visualize_3d_mesh(img_cv2, outputs, estimator.faces)
    
    # Display results
    for i, combined_img in enumerate(mesh_results):
        combined_rgb = cv2.cvtColor(combined_img, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(20, 5))
        plt.imshow(combined_rgb)
        plt.title(f'Person {i}: Original | Mesh Overlay | Front View | Side View')
        plt.axis('off')
        plt.show()
else:
    print("No people detected for 3D mesh visualization")

## 5. Save 3D Mesh Files and Results

In [ ]:
if outputs:
    # Get image name without extension
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    
    # Create output directory
    output_dir = f"output/{image_name}"

    # Save all results (PLY meshes, overlay images, bbox images)
    ply_files = save_mesh_results(img_cv2, outputs, estimator.faces, output_dir, image_name)
    
    print(f"\n=== Saved Results for {image_name} ===")
    print(f"Output directory: {output_dir}")
    print(f"Number of PLY files created: {len(ply_files)}")
    
else:
    print("No results to save - no people detected")

## 6. Mask-Based Inference

In [ ]:
# Load mask and run inference
mask_path = "images/dancing_mask.png"

if os.path.exists(mask_path):
    # Load and display the mask
    mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    plt.figure(figsize=(8, 6))
    plt.imshow(mask_img, cmap='gray')
    plt.title('External Mask')
    plt.axis('off')
    plt.show()
    
    # Process with external mask
    mask_outputs = process_image_with_mask(estimator, image_path, mask_path)
    
    # Visualize and save results
    if mask_outputs:
        mask_mesh_results = visualize_3d_mesh(img_cv2, mask_outputs, estimator.faces)
        
        for i, combined_img in enumerate(mask_mesh_results):
            combined_rgb = cv2.cvtColor(combined_img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(20, 5))
            plt.imshow(combined_rgb)
            plt.title(f'Mask-Based Person {i}: Original | Mesh Overlay | Front View | Side View')
            plt.axis('off')
            plt.show()
            
        # Save results
        mask_output_dir = f"output/mask_based_{image_name}"
        mask_ply_files = save_mesh_results(img_cv2, mask_outputs, estimator.faces, mask_output_dir, f"mask_{image_name}")
        print(f"Saved mask-based results to: {mask_output_dir}")
    else:
        print("No people detected with mask-based approach")
        
else:
    print(f"Mask file not found: {mask_path}")